In [10]:
import torch
import cv2
import numpy as np

import os
from extraction.images import model_loader
from extraction.video_processing import extract_video_features_compressed_ms

import time

In [11]:
violence_dir = ['data/images/violence/Violent', 'data/images/RWF-2000/Fight']
non_violence_dir = ['data/images/violence/Normal', 'data/images/RWF-2000/NonFight']

In [12]:
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

In [13]:
# 1. Device Guard Setup
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"⏳ Loading transformer weights into active hardware storage space...")

# 2. Initialize weights ONCE at the top level of the cell
global_model, global_processor, active_code = model_loader(model_code='clip', device=device)
print(f"✅ Transformer successfully cached on: {device.upper()}\n")

⏳ Loading transformer weights into active hardware storage space...
✅ Transformer successfully cached on: MPS



In [16]:
non_violent_data = []

# ─── PROCESS VIOLENT DATASET TRACK ───
for video_dir in non_violence_dir:
    if not os.path.isdir(video_dir): continue
    
    for video_file in os.listdir(video_dir):
        # Skip system garbage metadata files like .DS_Store
        if video_file.startswith('.'): continue 

        video_path = os.path.join(video_dir, video_file)      
        start_timer = time.time()
        
        # 🔥 FIXED: Passing global_model and global_processor variables smoothly instead of string codes
        vector = extract_video_features_compressed_ms(
            video_path=video_path,
            model=global_model,
            processor=global_processor,
            model_code=active_code,
            target_frames=18
        )
        
        non_violent_data.append(vector)
        execution_speed = time.time() - start_timer

In [17]:
from sklearn.model_selection import train_test_split
from training.kfold_train import stratified_kfold_train_val

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

from sklearn.metrics import confusion_matrix, classification_report

In [18]:
violent_labels = np.ones(len(violent_data))
non_violent_labels = np.zeros(len(non_violent_data))

violent_data = np.array(violent_data)
non_violent_data = np.array(non_violent_data)

In [19]:
x = np.concat([violent_data, non_violent_data])
y = np.concat([violent_labels, non_violent_labels])

In [20]:
x_train, x_test, y_train, y_test = train_test_split(x, y,
                                                    test_size=0.2)

In [30]:
log_reg = LogisticRegression(
    penalty='l2',          # Use L2 (Ridge) regularization
    C=0.5,                 # Slightly stronger regularization than default
    solver='liblinear',        # Standard efficient solver
    max_iter=1000,         # Increased to guarantee optimization convergence
    random_state=42,
    class_weight='balanced'
)

stratified_kfold_train_val(5,
                           0.26,
                           log_reg,
                           x_train,
                           y_train)

Starting 5-Fold Stratified Cross-Validation...

sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.7626 (When flagged positive, accuracy is 76.26%)
Custom Recall Score:    0.9152 (Captured 91.52% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.8270 (When flagged positive, accuracy is 82.70%)
Custom Recall Score:    0.9273 (Captured 92.73% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.7795 (When flagged positive, accuracy is 77.95%)
Custom Recall Score:    0.9212 (Captured 92.12% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.7684 (When flagged positive, accuracy is 76.84%)
Custom Recall Score:    0.8848 (Captured 88.48% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.7313 (When flagged positive, accuracy is 73.13%)
Custom R

In [41]:
log_reg = LogisticRegression(
    penalty='l2',          # Use L2 (Ridge) regularization
    C=0.5,                 # Slightly stronger regularization than default
    solver='liblinear',        # Standard efficient solver
    max_iter=1000,         # Increased to guarantee optimization convergence
    random_state=42,
    class_weight='balanced'
)
threshold = 0.26

log_reg.fit(x_train, y_train)
test_probabilities = log_reg.predict_proba(x_test)[:, 1]
predictions = (test_probabilities >= threshold).astype(int)

print(classification_report(y_test, predictions))
print(confusion_matrix(y_test, predictions))

              precision    recall  f1-score   support

         0.0       0.93      0.72      0.81       194
         1.0       0.79      0.95      0.86       218

    accuracy                           0.84       412
   macro avg       0.86      0.83      0.84       412
weighted avg       0.85      0.84      0.84       412

[[139  55]
 [ 11 207]]


In [42]:
np.savez("data/images/violence.npz", features=x, labels=y)

In [44]:
from xgboost import XGBClassifier

In [52]:
# 2. Instantiate the XGBoost Classifier with parameters from the documentation
xgb_model = XGBClassifier(
    # Core Parameters
    n_estimators=100,       
    learning_rate=0.1,      
    max_depth=6,            
    
    # Learning Task Parameters
    objective='binary:logistic',  
    eval_metric='logloss',        
    
    # Regularization / Overfitting Control
    subsample=0.8,          
    colsample_bytree=0.8,   # Subsample ratio of columns when constructing each tree
    gamma=0,                # Minimum loss reduction required to make a further partition
    reg_alpha=0,            # L1 regularization term on weights (Lasso)
    reg_lambda=1,           # L2 regularization term on weights (Ridge)
    
    # Hardware / Environment
    n_jobs=-1,              # Number of parallel threads used to run xgboost (-1 uses all cores)
    random_state=42,        # Random number seed
    verbosity=1             # Verbosity of printing messages (0 = silent, 1 = warning, 2 = info)
)

stratified_kfold_train_val(5,
                           0.23,
                           xgb_model,
                           x_train,
                           y_train)

Starting 5-Fold Stratified Cross-Validation...

sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.7680 (When flagged positive, accuracy is 76.80%)
Custom Recall Score:    0.9030 (Captured 90.30% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.8075 (When flagged positive, accuracy is 80.75%)
Custom Recall Score:    0.9152 (Captured 91.52% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.7937 (When flagged positive, accuracy is 79.37%)
Custom Recall Score:    0.9091 (Captured 90.91% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.7641 (When flagged positive, accuracy is 76.41%)
Custom Recall Score:    0.9030 (Captured 90.30% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.7641 (When flagged positive, accuracy is 76.41%)
Custom R

In [54]:
xgb_model = XGBClassifier(
    # Core Parameters
    n_estimators=100,       
    learning_rate=0.1,      
    max_depth=6,            
    
    # Learning Task Parameters
    objective='binary:logistic',  
    eval_metric='logloss',        
    
    # Regularization / Overfitting Control
    subsample=0.8,          
    colsample_bytree=0.8,   # Subsample ratio of columns when constructing each tree
    gamma=0,                # Minimum loss reduction required to make a further partition
    reg_alpha=0,            # L1 regularization term on weights (Lasso)
    reg_lambda=1,           # L2 regularization term on weights (Ridge)
    
    # Hardware / Environment
    n_jobs=-1,              # Number of parallel threads used to run xgboost (-1 uses all cores)
    random_state=42,        # Random number seed
    verbosity=1             # Verbosity of printing messages (0 = silent, 1 = warning, 2 = info)
)
threshold = 0.23

xgb_model.fit(x_train, y_train)
test_probabilities = xgb_model.predict_proba(x_test)[:, 1]
predictions = (test_probabilities >= threshold).astype(int)

print(classification_report(y_test, predictions))
print(confusion_matrix(y_test, predictions))

              precision    recall  f1-score   support

         0.0       0.92      0.69      0.79       194
         1.0       0.77      0.95      0.85       218

    accuracy                           0.83       412
   macro avg       0.85      0.82      0.82       412
weighted avg       0.84      0.83      0.82       412

[[133  61]
 [ 11 207]]


In [56]:
import joblib

joblib.dump(log_reg, open("modelling/model/violence_new.jobllib", 'wb'))